In [2]:
# ===== 방식 B: 공유 PCA + Batch별 임계값 (Hybrid) =====
# 완전 독립 실행 가능 코드

import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

# Step 0: 데이터 로드 (이미 했어도 다시)
uf = pd.read_csv('unified_features.csv', index_col=0)
uf.index = uf.index.astype(int)
uf['fault_name'] = uf['fault_name'].str.strip()

def get_batch(wid):
    if wid < 3000: return 'Batch_29'
    elif wid < 3200: return 'Batch_31'
    else: return 'Batch_33'

uf['batch'] = uf.index.map(get_batch)

# Step 1: X와 라벨 분리
X_all = uf.drop(columns=['fault_name', 'batch']).values
is_normal = (uf['fault_name'] == 'calibration').values
is_fault = ~is_normal

print(f"전체 데이터: {X_all.shape}")
print(f"정상: {is_normal.sum()}장, Fault: {is_fault.sum()}장\n")

# Step 2: Q 통계량 함수 정의
def Q_stat(X, model):
    X_recon = model.inverse_transform(model.transform(X))
    return ((X - X_recon)**2).sum(axis=1)

# Step 3: 공유 PCA 학습 (전체 정상 103장으로)
X_normal = X_all[is_normal]
pca_shared = PCA(n_components=0.90).fit(X_normal)
print(f"공유 PCA 주성분 수: {pca_shared.n_components_}")

# Step 4: 전체 데이터에 Q 계산
Q_all = Q_stat(X_all, pca_shared)

# Step 5: Batch별 임계값 산출
print("\n=== Batch별 임계값 (각 batch 정상의 99% 분위수) ===")
batch_thresholds = {}
for b in ['Batch_29','Batch_31','Batch_33']:
    mask_b = (uf['batch']==b).values & is_normal
    batch_thresholds[b] = np.quantile(Q_all[mask_b], 0.99)
    print(f"  {b} (n={mask_b.sum()}): 임계값 = {batch_thresholds[b]:.2f}")

# 비교용: 공통 임계값
global_threshold = np.quantile(Q_all[is_normal], 0.99)
print(f"\n(참고) 공통 임계값: {global_threshold:.2f}")

# Step 6: 각 웨이퍼는 자기 batch의 임계값으로 판정
hit_hybrid = np.zeros(len(X_all), dtype=bool)
for b in ['Batch_29','Batch_31','Batch_33']:
    mask_b = (uf['batch']==b).values
    hit_hybrid[mask_b] = Q_all[mask_b] > batch_thresholds[b]

# 비교용: 공통 임계값 판정
hit_global = Q_all > global_threshold

# Step 7: 평가
print("\n" + "="*60)
print("방식 B (Hybrid) 평가")
print("="*60)

tp = (hit_hybrid & is_fault).sum()
fn = (~hit_hybrid & is_fault).sum()
fp = (hit_hybrid & is_normal).sum()
tn = (~hit_hybrid & is_normal).sum()

print(f"\n전체 결과:")
print(f"  Fault 탐지: {tp}/{is_fault.sum()} (Recall {tp/is_fault.sum()*100:.1f}%)")
print(f"  False Positive: {fp}/{is_normal.sum()} (FAR {fp/is_normal.sum()*100:.2f}%)")
print(f"  Precision: {tp/(tp+fp)*100:.1f}%")

print(f"\nBatch별 FAR 비교:")
print(f"{'Batch':<12} {'기존(공통)':<15} {'방식 B (로컬)':<15}")
for b in ['Batch_29','Batch_31','Batch_33']:
    mask_b = (uf['batch']==b).values & is_normal
    far_global = hit_global[mask_b].mean() * 100
    far_hybrid = hit_hybrid[mask_b].mean() * 100
    print(f"  {b:<10} {far_global:>6.1f}%       {far_hybrid:>6.1f}%")

# Step 8: 통계 검정 - Chi-square로 FAR 균등성 검증
from scipy.stats import chi2_contingency

print(f"\n=== Chi-square: Batch별 FAR 균등성 ===")

# 기존 방식 (공통 임계값)
contingency_global = []
for b in ['Batch_29','Batch_31','Batch_33']:
    mask_b = (uf['batch']==b).values & is_normal
    fp_b = hit_global[mask_b].sum()
    tn_b = mask_b.sum() - fp_b
    contingency_global.append([fp_b, tn_b])

# 방식 B
contingency_hybrid = []
for b in ['Batch_29','Batch_31','Batch_33']:
    mask_b = (uf['batch']==b).values & is_normal
    fp_b = hit_hybrid[mask_b].sum()
    tn_b = mask_b.sum() - fp_b
    contingency_hybrid.append([fp_b, tn_b])

_, p_global, _, _ = chi2_contingency(np.array(contingency_global))
_, p_hybrid, _, _ = chi2_contingency(np.array(contingency_hybrid))

print(f"  기존 (공통 임계값): p = {p_global:.4f}")
print(f"  방식 B (로컬 임계값): p = {p_hybrid:.4f}")
print(f"  → p값이 클수록 batch 간 FAR이 균등하다는 의미")

전체 데이터: (123, 8478)
정상: 103장, Fault: 20장

공유 PCA 주성분 수: 69

=== Batch별 임계값 (각 batch 정상의 99% 분위수) ===
  Batch_29 (n=32): 임계값 = 1926.98
  Batch_31 (n=36): 임계값 = 1698.24
  Batch_33 (n=35): 임계값 = 1264.41

(참고) 공통 임계값: 1911.17

방식 B (Hybrid) 평가

전체 결과:
  Fault 탐지: 20/20 (Recall 100.0%)
  False Positive: 3/103 (FAR 2.91%)
  Precision: 87.0%

Batch별 FAR 비교:
Batch        기존(공통)          방식 B (로컬)      
  Batch_29      6.2%          3.1%
  Batch_31      0.0%          2.8%
  Batch_33      0.0%          2.9%

=== Chi-square: Batch별 FAR 균등성 ===
  기존 (공통 임계값): p = 0.1041
  방식 B (로컬 임계값): p = 0.9961
  → p값이 클수록 batch 간 FAR이 균등하다는 의미
